In [1]:
import pandas as pd
import numpy as np
from sgp4.api import Satrec, jday
from datetime import datetime, timezone

tle = pd.read_parquet("/home/darshani/lightkurve-env/space-debris-detector/data/cleaned/tle_cleaned.parquet")
print(tle.shape)
tle[['NORAD_CAT_ID', 'OBJECT_NAME', 'TLE_LINE1', 'TLE_LINE2', 'EPOCH']].head(10)

(66727, 40)


,NORAD_CAT_ID,OBJECT_NAME,TLE_LINE1,TLE_LINE2,EPOCH
0,4,EXPLORER 1,1 00004U 58001 A 70090.03500497 .07718844 +0...,2 00004 033.1468 334.6171 0024739 311.5310 048...,1970-03-31T00:50:24.429408
1,5,VANGUARD 1,1 00005U 58002B 26079.71544731 .00000108 0...,2 00005 34.2414 227.3506 1836943 99.5572 281...,2026-03-20T17:10:14.647584
2,8,SPUTNIK 3,1 00008U 58004 B 60095.16166626 .02607090 +0...,2 00008 065.0599 163.5585 0088318 135.6760 224...,1960-04-04T03:52:47.964864
3,9,EXPLORER 4,1 00009U 58005 A 59275.28696823 .00892850 +0...,2 00009 050.2549 082.4216 0254534 057.7226 303...,1959-10-02T06:53:14.055072
4,11,VANGUARD 2,1 00011U 59001A 26080.38145253 .00000908 0...,2 00011 32.8774 57.4304 1445707 277.1980 66...,2026-03-21T09:09:17.498592
5,12,VANGUARD R/B,1 00012U 59001B 26080.42463896 .00000465 0...,2 00012 32.8980 174.5409 1646212 0.3654 359...,2026-03-21T10:11:28.806144
6,15,EXPLORER 6,1 00015U 59004 A 59272.62543869 .00008106 +0...,2 00015 046.9509 049.6823 7601812 047.2577 341...,1959-09-29T15:00:37.902816
7,16,VANGUARD R/B,1 00016U 58002A 26080.61892694 -.00000001 0...,2 00016 34.2588 198.2539 2019906 110.7408 272...,2026-03-21T14:51:15.287616
8,17,THOR ABLE R/B,1 00017U 59004 B 61008.85131000 .00070225 +0...,2 00017 047.1000 284.3200 7526400 171.0000 203...,1961-01-08T20:25:53.183999
9,18,DISCOVERER 5,1 00018U 59005 A 59269.32477950 .04655228 +0...,2 00018 079.9899 251.4819 0140209 012.7649 347...,1959-09-26T07:47:40.948799


In [2]:
tle['EPOCH'] = pd.to_datetime(tle['EPOCH'])
now = datetime.now(timezone.utc)
tle['tle_age_days'] = (now - tle['EPOCH'].dt.tz_localize('UTC')).dt.days

tle['PERIAPSIS'] = pd.to_numeric(tle['PERIAPSIS'], errors='coerce')
usable = tle[~((tle['tle_age_days'] > 30) & (tle['PERIAPSIS'] < 200))].copy()

print("running sgp4 on", len(usable), "objects")
usable[['NORAD_CAT_ID', 'OBJECT_NAME', 'TLE_LINE1', 'TLE_LINE2', 'tle_age_days']].head()

running sgp4 on 50146 objects


,NORAD_CAT_ID,OBJECT_NAME,TLE_LINE1,TLE_LINE2,tle_age_days
1,5,VANGUARD 1,1 00005U 58002B 26079.71544731 .00000108 0...,2 00005 34.2414 227.3506 1836943 99.5572 281...,1
3,9,EXPLORER 4,1 00009U 58005 A 59275.28696823 .00892850 +0...,2 00009 050.2549 082.4216 0254534 057.7226 303...,24277
4,11,VANGUARD 2,1 00011U 59001A 26080.38145253 .00000908 0...,2 00011 32.8774 57.4304 1445707 277.1980 66...,0
5,12,VANGUARD R/B,1 00012U 59001B 26080.42463896 .00000465 0...,2 00012 32.8980 174.5409 1646212 0.3654 359...,0
6,15,EXPLORER 6,1 00015U 59004 A 59272.62543869 .00008106 +0...,2 00015 046.9509 049.6823 7601812 047.2577 341...,24280


In [3]:
jd, fr = jday(now.year, now.month, now.day, now.hour, now.minute, now.second)

rows = []
for i, row in usable.iterrows():
    try:
        sat = Satrec.twoline2rv(row['TLE_LINE1'], row['TLE_LINE2'])
        err, r, v = sat.sgp4(jd, fr)
        rows.append({
            'NORAD_CAT_ID': row['NORAD_CAT_ID'],
            'error': err,
            'rx_km': r[0], 'ry_km': r[1], 'rz_km': r[2],
            'vx_km_s': v[0], 'vy_km_s': v[1], 'vz_km_s': v[2],
            'tle_age_days': row['tle_age_days']
        })
    except Exception as e:
        rows.append({
            'NORAD_CAT_ID': row['NORAD_CAT_ID'],
            'error': 999,
            'rx_km': None, 'ry_km': None, 'rz_km': None,
            'vx_km_s': None, 'vy_km_s': None, 'vz_km_s': None,
            'tle_age_days': row['tle_age_days']
        })

sgp4_df = pd.DataFrame(rows)
print(sgp4_df['error'].value_counts())

error
0    37892
1    11583
6      662
4        8
3        1
Name: count, dtype: int64


In [4]:
sgp4_df = sgp4_df[sgp4_df['error'] == 0].copy()

sgp4_df['altitude_km'] = np.sqrt(sgp4_df['rx_km']**2 + sgp4_df['ry_km']**2 + sgp4_df['rz_km']**2) - 6371
sgp4_df['speed_km_s'] = np.sqrt(sgp4_df['vx_km_s']**2 + sgp4_df['vy_km_s']**2 + sgp4_df['vz_km_s']**2)

sgp4_df['regime'] = pd.cut(sgp4_df['altitude_km'],
    bins=[0, 450, 2000, 35786, 1000000],
    labels=['VLEO', 'LEO', 'MEO', 'GEO'])

print(sgp4_df['regime'].value_counts())
sgp4_df[['altitude_km', 'speed_km_s']].describe()

regime
LEO     26423
VLEO     4077
MEO      4054
GEO      1879
Name: count, dtype: int64


,altitude_km,speed_km_s
count,3.789200e+04,3.789200e+04
mean,9.605447e+30,3.283171e+02
std,1.202445e+33,3.498098e+04
min,1.239555e+01,1.446971e-15
25%,4.923137e+02,7.177907e+00
50%,7.620007e+02,7.487680e+00
75%,1.387666e+03,7.619829e+00
max,1.991883e+35,5.379635e+06


In [5]:
sgp4_df.to_parquet("/home/darshani/lightkurve-env/space-debris-detector/data/sgp4_positions.parquet", index=False)
print("saved!", sgp4_df.shape)
sgp4_df.head()

saved! (37892, 12)


,NORAD_CAT_ID,error,rx_km,ry_km,rz_km,vx_km_s,vy_km_s,vz_km_s,tle_age_days,altitude_km,speed_km_s,regime
0,5,0,-8210.998356,-3990.498411,-1923.840662,2.574536,-4.548402,3.446308,1,2958.830356,6.260450,MEO
1,9,0,-3519.404089,3015.901726,-4733.148996,-6.414072,-3.952547,2.200901,24277,253.542836,7.849007,VLEO
2,11,0,-1388.811012,7840.407708,3589.239802,-5.787650,-1.342493,2.607252,0,2363.038666,6.488216,MEO
3,12,0,9335.617162,-2249.540767,726.537695,1.601603,4.717411,-3.154739,0,3259.266791,5.896734,MEO
5,16,0,-7531.439043,-4180.168420,1152.946477,4.324416,-3.969845,3.472876,0,2319.550481,6.820639,MEO
